In [7]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [8]:
df=pd.read_csv('/content/drive/MyDrive/airline_faq.csv')
df

,Question,Answer
0,Can I get a refund if I cancel my Aurora Skies...,"Yes, Aurora Skies Airways allows refunds withi..."
1,What happens if Aurora Skies Airways changes m...,If Aurora Skies Airways changes your flight sc...
2,Are change or cancellation fees applicable to ...,Change or cancellation fees may apply based on...
3,How can I modify my Aurora Skies Airways booking?,You can access your booking online to modify y...
4,What are my options if my Aurora Skies Airways...,"In such cases, Aurora Skies Airways offers the..."
5,Does Aurora Skies Airways charge a refund proc...,"No, Aurora Skies Airways has never charged a r..."
6,What should I do if I need to rebook my flight...,If your flight is delayed or canceled by Auror...
7,Are taxes and fees refundable on unused tickets?,Refundability of taxes and fees depends on the...
8,Can I retain the value of my ticket for future...,"Yes, to retain the value of your ticket for fu..."
9,How does Aurora Skies Airways handle schedule ...,Aurora Skies Airways has established guideline...


In [9]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
df["embedding"] = df["Question"].apply(lambda x: embedder.encode(x))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [11]:
def retrieve_top_k(query, df, top_k=3):
    query_vec = embedder.encode(query)
    similarities = df["embedding"].apply(lambda x: cosine_similarity([query_vec], [x])[0][0])
    top_docs = df.iloc[np.argsort(similarities)[-top_k:][::-1]]
    return top_docs

In [12]:
def build_prompt(query, retrieved_df):
    context = "\n".join(
        [f"Q: {row['Question']}\nA: {row['Answer']}" for _, row in retrieved_df.iterrows()]
    )
    prompt = f"""
You are Aurora Airlines' official customer support assistant.

Use *only* the provided information to answer the question below.
If the information is not present, reply:
"I’m sorry, I don’t have that information in Aurora Airlines' current policy."

Your answer must start with “Aurora Airlines” and be 2–3 lines long.

Context:
{context}

Question: {query}
Answer:
"""
    return prompt

In [13]:
def ask_t5(prompt, max_length=200):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_length=max_length, temperature=0.2, num_beams=3)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [14]:
def validate_answer(answer, retrieved_df):
    context_text = " ".join(retrieved_df["Answer"].tolist()).lower()
    overlap = sum(word in context_text for word in answer.lower().split())
    if overlap < 3:
        return "I’m sorry, I don’t have that information in Aurora Airlines' current policy."
    return answer

In [15]:
query = "Can I rebook my Aurora Skies Airways flight online if it was changed by the airline?"

retrieved = retrieve_top_k(query, df)
prompt = build_prompt(query, retrieved)
raw_answer = ask_t5(prompt)
final_answer = validate_answer(raw_answer, retrieved)

print("User Query:", query)
print("Final Answer:\n", final_answer)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


User Query: Can I rebook my Aurora Skies Airways flight online if it was changed by the airline?
Final Answer:
 Yes, Aurora Skies Airways allows refunds within 24 hours of purchase for all fare types, including published and net fares, as well as tickets with codeshare and interline flights. This policy does not apply to group fares or fares purchased for same-day travel.


In [16]:
query = "What if I miss my connecting flight?"

retrieved = retrieve_top_k(query, df)
prompt = build_prompt(query, retrieved)
raw_answer = ask_t5(prompt)
final_answer = validate_answer(raw_answer, retrieved)

print("User Query:", query)
print("Final Answer:\n", final_answer)

User Query: What if I miss my connecting flight?
Final Answer:
 I’m sorry, I don’t have that information in Aurora Airlines’ current policy.
